# Student Performance Early Warning System

**Authors:** Gutenberg Petit-vil, Destiny Milfort
**Course:** CAP4630 — Introduction to Artificial Intelligence
**Track:** Research

This notebook investigates *how early in the semester we can reliably identify at-risk students*, using the UCI Student Performance dataset (Cortez & Silva, 2008). Three classifiers — logistic regression, decision tree, random forest — are compared under three feature regimes simulating data availability at the start of term, after the first grading period, and at end-of-term. Each model is benchmarked against a simple rule-based baseline, and SHAP feature attribution is used to identify which signals drive early predictions and whether they are *actionable* (study habits, absences) or *static* (parental education, demographics).

**Three experiments:**
1. Does ML beat a common-sense baseline?
2. How does prediction degrade as we predict earlier?
3. What drives early predictions, and are those features actionable?


## 1. Setup

In [ ]:
# Install SHAP (the only non-default Colab dependency)
!pip install -q shap

In [ ]:
import os
import urllib.request
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the UCI Student Performance Dataset

The dataset (Cortez & Silva, 2008) records demographic, family, behavioral, and academic features for ~395 Portuguese secondary school students taking a Math course. We use `student-mat.csv`. Key columns for our analysis:

- `G1`, `G2`, `G3` — first-period, second-period, and final grades (0–20 scale)
- `failures` — number of past class failures (used by our rule-based baseline)
- 30 other demographic, family, and behavioral features

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def _extract_recursively(zip_path, dest="."):
    """Extract zip; if it contains nested zips, extract those too."""
    import glob
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    for inner in glob.glob(os.path.join(dest, "*.zip")):
        if os.path.basename(inner) == os.path.basename(zip_path):
            continue
        with zipfile.ZipFile(inner) as z:
            z.extractall(dest)

if not os.path.exists("student-mat.csv"):
    urllib.request.urlretrieve(URL, "outer.zip")
    _extract_recursively("outer.zip")

assert os.path.exists("student-mat.csv"), "Download/extract failed; check the UCI URL."

df = pd.read_csv("student-mat.csv", sep=";")
print("Shape:", df.shape)
df.head()

## 3. Define the Target

We define *at-risk* as a final grade below the Portuguese passing threshold of 10:

```
at_risk = 1 if G3 < 10 else 0
```

In [ ]:
df["at_risk"] = (df["G3"] < 10).astype(int)
print(f"At-risk rate: {df['at_risk'].mean():.1%}  ({df['at_risk'].sum()} of {len(df)} students)")
df["at_risk"].value_counts()

## 4. Build the Three Earliness Regimes

We construct three feature sets to simulate three points in the semester:

| Regime | When | Features |
|---|---|---|
| **Early** | Start of term | Demographic + family + behavioral (no grades) |
| **Mid**   | After first grading period | Early features **+ G1** |
| **Late**  | End of term | Early features **+ G1 + G2** |

The same train/test indices are used across all three regimes so the comparison is apples-to-apples.

In [ ]:
# All features except the final grade and our derived target
all_features = [c for c in df.columns if c not in ["G3", "at_risk"]]
late_features  = list(all_features)
mid_features   = [c for c in all_features if c != "G2"]
early_features = [c for c in all_features if c not in ["G1", "G2"]]

print(f"Late  ({len(late_features)} features):  ...{late_features[-3:]}")
print(f"Mid   ({len(mid_features)} features):   ...{mid_features[-3:]}")
print(f"Early ({len(early_features)} features): ...{early_features[-3:]}")

In [ ]:
def encode(frame, features):
    """Label-encode object columns; return numeric DataFrame."""
    X = frame[features].copy()
    for col in X.select_dtypes(include="object").columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    return X

X_late  = encode(df, late_features)
X_mid   = encode(df, mid_features)
X_early = encode(df, early_features)
y = df["at_risk"]

# Stratified split, fixed indices reused across regimes
idx_train, idx_test = train_test_split(
    df.index, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
y_train, y_test = y.loc[idx_train], y.loc[idx_test]
print(f"Train: {len(idx_train)}   Test: {len(idx_test)}   At-risk in test: {y_test.sum()}")

## 5. Evaluation Helper

In [ ]:
def evaluate(y_true, y_pred, y_proba=None):
    out = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        try:
            out["ROC-AUC"] = roc_auc_score(y_true, y_proba)
        except Exception:
            out["ROC-AUC"] = float("nan")
    return out

def fit_predict(model_name, X_train, y_train, X_test):
    """Returns (predictions, probabilities, fitted_model)."""
    if model_name == "Logistic Regression":
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        m = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
        m.fit(Xtr, y_train)
        return m.predict(Xte), m.predict_proba(Xte)[:, 1], m
    elif model_name == "Decision Tree":
        m = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
        m.fit(X_train, y_train)
        return m.predict(X_test), m.predict_proba(X_test)[:, 1], m
    elif model_name == "Random Forest":
        m = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
        m.fit(X_train, y_train)
        return m.predict(X_test), m.predict_proba(X_test)[:, 1], m
    raise ValueError(model_name)

MODELS = ["Logistic Regression", "Decision Tree", "Random Forest"]

## Experiment 1 — Does ML beat a common-sense baseline?

The baseline rule: **flag any student with at least one previous course failure.** This is a one-line predictor a school administrator could apply by hand. We train all three models on the *Late* (end-of-term) regime and check whether they meaningfully beat this rule.

> If the baseline ties the models, the rest of the analysis is moot. This sanity check is missing from most basic ML projects.

In [ ]:
results_exp1 = {}

# Rule-based baseline
baseline_pred = (df.loc[idx_test, "failures"] > 0).astype(int).values
results_exp1["Baseline (failures>0)"] = evaluate(y_test, baseline_pred)

# Three ML models on the Late regime
for name in MODELS:
    pred, proba, _ = fit_predict(name, X_late.loc[idx_train], y_train, X_late.loc[idx_test])
    results_exp1[name] = evaluate(y_test, pred, proba)

exp1_df = pd.DataFrame(results_exp1).T.round(3)
exp1_df

## Experiment 2 — How does prediction degrade with earliness?

We train each model on each regime and plot recall and F1 as a function of how early the prediction is made.

> Hypothesis: recall drops sharply between *Mid* and *Early* once G1 is removed, indicating that the strongest predictive signal is simply the most recent grade.

In [ ]:
regime_data = {
    "Early (no grades)": X_early,
    "Mid (G1 only)":     X_mid,
    "Late (G1+G2)":      X_late,
}

rows = []
trained = {}  # save for SHAP later
for regime_name, X_full in regime_data.items():
    Xtr_full = X_full.loc[idx_train]
    Xte_full = X_full.loc[idx_test]
    for model_name in MODELS:
        pred, proba, model = fit_predict(model_name, Xtr_full, y_train, Xte_full)
        metrics = evaluate(y_test, pred, proba)
        metrics["Regime"] = regime_name
        metrics["Model"]  = model_name
        rows.append(metrics)
        trained[(regime_name, model_name)] = model

exp2_df = pd.DataFrame(rows)
exp2_df.round(3)

In [ ]:
# Pivot tables for the headline numbers
print("=== Recall by Regime x Model ===")
print(exp2_df.pivot(index="Regime", columns="Model", values="Recall").round(3))
print()
print("=== F1 by Regime x Model ===")
print(exp2_df.pivot(index="Regime", columns="Model", values="F1").round(3))

In [ ]:
regime_order = ["Early (no grades)", "Mid (G1 only)", "Late (G1+G2)"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for metric, ax in zip(["Recall", "F1"], axes):
    pivot = (
        exp2_df.pivot(index="Regime", columns="Model", values=metric)
              .reindex(regime_order)
    )
    pivot.plot(marker="o", ax=ax, linewidth=2, markersize=8)
    ax.set_title(f"{metric} vs. Earliness", fontsize=13)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")
plt.suptitle("How prediction quality changes as we move earlier in the semester", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Experiment 3 — What drives early predictions, and are those features actionable?

We compute SHAP values for the random forest under the *Early* regime, rank features by mean absolute SHAP, and tag each as **actionable** (something a school could intervene on — study time, absences, going out) or **static** (something the school cannot change — sex, parental education, address).

> Hypothesis: the top features will be dominated by static demographic variables, suggesting that an early-warning system trained on this data is partly a demographic predictor in disguise.

In [ ]:
import shap

rf_early = trained[("Early (no grades)", "Random Forest")]
X_early_test = X_early.loc[idx_test]

explainer = shap.TreeExplainer(rf_early)
shap_raw = explainer.shap_values(X_early_test)

# SHAP returns slightly different shapes depending on version; normalize to 2D for class 1
if isinstance(shap_raw, list):
    shap_at_risk = shap_raw[1]
elif hasattr(shap_raw, "ndim") and shap_raw.ndim == 3:
    shap_at_risk = shap_raw[:, :, 1]
else:
    shap_at_risk = shap_raw

print("SHAP matrix shape:", shap_at_risk.shape)

In [ ]:
# Bar summary: top features by mean absolute SHAP
shap.summary_plot(shap_at_risk, X_early_test, plot_type="bar", show=False, max_display=12)
plt.title("Top features driving the EARLY at-risk prediction", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Tag each feature as actionable or static
ACTIONABLE = {
    "studytime", "absences", "goout", "Dalc", "Walc", "failures",
    "freetime", "health", "romantic", "paid", "activities",
    "schoolsup", "famsup", "higher", "internet", "traveltime",
    "famrel",
}
STATIC = {
    "sex", "age", "school", "address", "famsize", "Pstatus",
    "Medu", "Fedu", "Mjob", "Fjob", "reason", "guardian", "nursery",
}

def tag(feature):
    if feature in ACTIONABLE: return "actionable"
    if feature in STATIC:     return "static"
    return "unknown"

mean_abs = np.abs(shap_at_risk).mean(axis=0)
top10 = (
    pd.DataFrame({"feature": X_early.columns, "mean_abs_shap": mean_abs})
      .sort_values("mean_abs_shap", ascending=False)
      .head(10)
      .reset_index(drop=True)
)
top10["type"] = top10["feature"].apply(tag)
top10

In [ ]:
# Summary count: how many of the top 10 are static vs actionable?
counts = top10["type"].value_counts()
print("Top 10 early-prediction features by category:")
print(counts.to_string())
print()
n_static = counts.get("static", 0)
n_action = counts.get("actionable", 0)
if n_static >= n_action:
    print(f"=> Static features ({n_static}) outweigh actionable ones ({n_action}).")
    print("   An early-warning system from this data leans demographic.")
else:
    print(f"=> Actionable features ({n_action}) outweigh static ones ({n_static}).")
    print("   An early-warning system from this data points to behaviors a school can address.")

## Findings (fill in after running)

**Experiment 1 — ML vs. baseline.** [Compare F1 of best model vs. baseline. Is the gap meaningful?]

**Experiment 2 — Earliness/accuracy trade-off.** [Where does recall drop the most: Late→Mid or Mid→Early? Which model degrades most gracefully?]

**Experiment 3 — Actionable vs. static drivers.** [How many of the top 10 early-regime features are static? What does that imply for using this kind of model in practice?]

**Limitations.**
- Single dataset, single course (Math); replicate on `student-por.csv` to check.
- ~395 students is small; results have wide confidence intervals.
- "At-risk" defined by a hard cutoff at G3 < 10; a regression view might tell a different story.
